# Cluster number commutator cost using up to 4-rdm

In [1]:
import numpy as np

# Molecule input parameters
molecule = 'h2o'              # supports: h2, h2o, n2, lih, h4_linear, h4_square
bond_length = 1.            # Angstrom
basis_set = 'sto3g'              # 'sto3g', '6-31g', ...
cluster_matrix = np.array([
        [1, 1, 0, 0, 0, 0, 0],
        [0, 0, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 1, 1, 0]
    ])

# DMRG parameters
bond_dim = 50 # 500
n_sweeps = 10 # 20

In [2]:
# Verify input validity

# Calculate the sum of each column (one column per orbital)
column_sums = np.sum(cluster_matrix, axis=0)
if not np.all(np.isin(column_sums, [0, 1])):
    invalid_columns = np.where((column_sums != 0) & (column_sums != 1))[0]
    raise ValueError(f"Error: Orbitals {invalid_columns} should appear in at most one cluster.")
if any([all(bit == 0 for bit in cluster) for cluster in cluster_matrix]):
    raise ValueError(f"Error: There is one or more empty clusters.")
# We do accept redundant clustering.
#    if np.all(column_sums == 1):
#        raise ValueError(f"Error: The clusters cover all orbitals. Remove one cluster to avoid redundancy.")
    num_clusters = len(cluster_matrix)

In [3]:
# ============================================================================
# SECTION 1: Get MOs and mps
# ============================================================================
import sys
from pathlib import Path
import hashlib
import json
sys.path.insert(0, '..') 
import src.cluster_number_operators as cluster_nums
import pyscf
from chemistry import get_geometry_and_description
from src.dmrg_solver import Block2DMRGSolver, DMRGConfig, solve_or_load_ground_state
from math import comb

# --- Step 1.1: Build molecule and run HF - we need an ONB and Hamiltonian ---
geometry, _ = get_geometry_and_description(molecule, bond_length)
mol = pyscf.M(atom=geometry, basis=basis_set)
mf = pyscf.scf.RHF(mol)
mf.kernel()

norb, nelec = mol.nao, mol.nelec
# assert norb == cluster_matrix.shape[1], f"Number of columns of cluster_matrix = {cluster_matrix.shape[1]} does not match number of orbitals = {norb}"
print(f"Number of orbitals: {norb}")
print(f"Number of elecs: {nelec}")
dim = comb(norb, nelec[0]) * comb(norb, nelec[1])
print(f"Hilbert space dimension: {dim}")
h1e = mf.mo_coeff.T @ mf.get_hcore() @ mf.mo_coeff
g2e = pyscf.ao2mo.full(mol, mf.mo_coeff) # compressed
g2e_full = pyscf.ao2mo.restore(1, g2e, norb) # not compressed; chemist's notation
# recall: chemist notation (pq|rs) = int d1 d2 \bar phi_p(1) phi_q(1) 1/r_12 \bar phi_r(2) phi_s(2) = <p, r| W |q, s>
ecore = mol.energy_nuc()

# --- Step 1.2: Initialize dummy DMRG object ---
solver = Block2DMRGSolver( # handles nelec as integer or pair
    h1e=h1e, g2e=g2e, ecore=ecore,
    n_elec=nelec, spin=mol.spin
)

# random mps
mps = solver.driver.get_random_mps(tag='RAND', bond_dim=5, nroots=1) # tensors are real valued

# fci state
fci_state = solver.to_ci_vector(ket=mps)

converged SCF energy = -74.9646625391309
Number of orbitals: 7
Number of elecs: (5, 5)
Hilbert space dimension: 441


In [4]:
# ============================================================================
# SECTION 2: Get 1, ..., 4-RDMs from the mps
# ============================================================================

# --- Step 2.1: Extract RDMs from MPS ---

# get rdms
rdm1_a, rdm1_b = solver.driver.get_1pdm(mps)
print("got 1rdm")
rdm2_aa, rdm2_ab, rdm2_bb = solver.driver.get_2pdm(mps)
print("got 2rdm")
rdm3_aaa, rdm3_aab, rdm3_abb, rdm3_bbb = solver.driver.get_3pdm(mps)
print("got 3rdm")
rdm4_aaaa, rdm4_aaab, rdm4_aabb, rdm4_abbb, rdm4_bbbb = solver.driver.get_4pdm(mps)
print("got 4rdm")

# --- Step 2.2: Spin-summed rdms ---

rdm1 = rdm1_a + rdm1_b
rdm2 = rdm2_aa + rdm2_bb + rdm2_ab + rdm2_ab.transpose(1, 0, 3, 2)

rdm3 = (
    rdm3_aaa 
    + rdm3_aab + rdm3_aab.transpose(0, 2, 1, 4, 3, 5) + rdm3_aab.transpose(2, 1, 0, 5, 4, 3) 
    + rdm3_abb + rdm3_abb.transpose(1, 0, 2, 3, 5, 4) + rdm3_abb.transpose(2, 1, 0, 5, 4, 3) 
    + rdm3_bbb
)

rdm4 = (
    # 0 betas (k=0)
    rdm4_aaaa
    
    # 1 beta (k=1)
    # Target beta positions: (3,), (2,), (1,), (0,)
    + rdm4_aaab
    + rdm4_aaab.transpose(0, 1, 3, 2,     5, 4, 6, 7) # (0, 1, 3, 2, 5, 4, 6, 7) 
    + rdm4_aaab.transpose(0, 3, 2, 1,     6, 5, 4, 7) # (0, 3, 1, 2, 5, 6, 4, 7) 
    + rdm4_aaab.transpose(3, 1, 2, 0,     7, 5, 6, 4) # (3, 0, 1, 2, 5, 6, 7, 4) 
    
    # 2 betas (k=2)
    # Target beta positions: (2,3), (1,3), (1,2), (0,3), (0,2), (0,1)
    + rdm4_aabb
    + rdm4_aabb.transpose(0, 2, 1, 3,     4, 6, 5, 7) # (0, 2, 1, 3, 4, 6, 5, 7) 
    + rdm4_aabb.transpose(0, 3, 2, 1,     6, 5, 4, 7) # (0, 2, 3, 1, 6, 4, 5, 7) 
    + rdm4_aabb.transpose(2, 1, 0, 3,     4, 7, 6, 5) # (2, 0, 1, 3, 4, 6, 7, 5) 
    + rdm4_aabb.transpose(3, 1, 2, 0,     7, 5, 6, 4) # (2, 0, 3, 1, 6, 4, 7, 5) 
    + rdm4_aabb.transpose(2, 3, 0, 1,     6, 7, 4, 5) # (2, 3, 0, 1, 6, 7, 4, 5)
    
    # 3 betas (k=3)
    # Target beta positions: (1,2,3), (0,2,3), (0,1,3), (0,1,2)
    + rdm4_abbb
    + rdm4_abbb.transpose(1, 0, 2, 3,     4, 5, 7, 6) # (1, 0, 2, 3, 4, 5, 7, 6) 
    + rdm4_abbb.transpose(2, 1, 0, 3,     4, 7, 6, 5) # (1, 2, 0, 3, 4, 7, 5, 6) 
    + rdm4_abbb.transpose(3, 1, 2, 0,     7, 5, 6, 4) # (1, 2, 3, 0, 7, 4, 5, 6) 
    
    # 4 betas (k=4)
    + rdm4_bbbb
)

got 1rdm
got 2rdm
got 3rdm
got 4rdm


In [5]:
# Do a basis rotation to rdms, fci_state, and electron integrals

from scipy.stats import unitary_group
import ffsim

U = unitary_group.rvs(dim=norb)
U_conj = np.conjugate(U)
# print((U @ U_conj.T).round(10))

rdm1_rotated = U_conj @ rdm1 @ U.T
rdm2_rotated = np.einsum('pi,qj,rk,sl,ijkl->pqrs', U_conj, U_conj, U, U, rdm2, optimize=True)
rdm3_rotated = np.einsum('pi,qj,rk,sl,tm,un,ijklmn->pqrstu', U_conj, U_conj, U_conj, U, U, U, rdm3, optimize=True)
rdm4_rotated = np.einsum('pi,qj,rk,sl,tm,un,vo,wa,ijklmnoa->pqrstuvw', U_conj, U_conj, U_conj, U_conj, U, U, U, U, rdm4, optimize=True)

fci_state_rotated = ffsim.apply_orbital_rotation(fci_state, U, norb, nelec)

h1e_rotated = np.einsum('pr,qs,rs->pq', U, U_conj, h1e, optimize=True)
g2e_full_rotated = np.einsum('pr,tu,qs,vz,rsuz->pqtv', U, U, U_conj, U_conj, g2e_full, optimize=True)

In [6]:
# --- Verify rotated 3RDM ---
import numpy as np
import itertools
import ffsim

# fermionic operators (alpha and beta)
cre_a, des_a = ffsim.cre_a, ffsim.des_a
cre_b, des_b = ffsim.cre_b, ffsim.des_b

# --- Verify 3RDM ---
orbitals = [3, 6, 1, 2]
for i, j, k, l, m, n in itertools.product(orbitals, repeat=6):
    expected = 0.0
    expected_rotated = 0.0
    # Sum over all spin combinations for the three creation operators
    for spin_i in [cre_a, cre_b]:
        for spin_j in [cre_a, cre_b]:
            for spin_k in [cre_a, cre_b]:
                # Determine the corresponding annihilation operators
                # For des(l): same spin as cre(i)
                des_l = des_a if spin_k == cre_a else des_b
                # For des(m): same spin as cre(j)
                des_m = des_a if spin_j == cre_a else des_b
                # For des(n): same spin as cre(k)
                des_n = des_a if spin_i == cre_a else des_b

                # Construct the operator: cre(i) cre(j) cre(k) des(l) des(m) des(n)
                op = ffsim.linear_operator(
                    ffsim.FermionOperator({
                        (spin_i(i), spin_j(j), spin_k(k), des_l(l), des_m(m), des_n(n)): 1
                    }),
                    norb, nelec
                )
                expected += np.vdot(fci_state, op @ fci_state)
                expected_rotated += np.vdot(fci_state_rotated, op @ fci_state_rotated)
    # Compare with your implemented rdm3
    if not np.isclose(rdm3[i, j, k, l, m, n], expected):
        print(f"3RDM mismatch at ({i},{j},{k},{l},{m},{n}): "
              f"Expected {expected}, Got {rdm3[i, j, k, l, m, n]}")
    if not np.isclose(rdm3_rotated[i, j, k, l, m, n], expected_rotated):
            print(f"Rotated 3RDM mismatch at ({i},{j},{k},{l},{m},{n}): "
                  f"Expected {expected_rotated}, Got {rdm3_rotated[i, j, k, l, m, n]}")


In [7]:
"""
# --- Verify 4RDM; skip, takes 15 min---
orbitals = [1, 2, 3, 4]
for i, j, k, l, m, n, p, q in itertools.product(orbitals, repeat=8):
    expected = 0.0
    expected_rotated = 0.0
    # Sum over all spin combinations for the four creation operators
    for spin_i in [cre_a, cre_b]:
        for spin_j in [cre_a, cre_b]:
            for spin_k in [cre_a, cre_b]:
                for spin_l in [cre_a, cre_b]:
                    # Determine the corresponding annihilation operators
                    des_m = des_a if spin_l == cre_a else des_b
                    des_n = des_a if spin_k == cre_a else des_b
                    des_p = des_a if spin_j == cre_a else des_b
                    des_q = des_a if spin_i == cre_a else des_b

                    # Construct the operator: cre(i) cre(j) cre(k) cre(l) des(q) des(p) des(n) des(m)
                    op = ffsim.linear_operator(
                        ffsim.FermionOperator({
                            (spin_i(i), spin_j(j), spin_k(k), spin_l(l),
                             des_m(m), des_n(n), des_p(p), des_q(q)): 1
                        }),
                        norb, nelec
                    )
                    expected += np.vdot(fci_state, op @ fci_state)
                    expected_rotated += np.vdot(fci_state_rotated, op @ fci_state_rotated)
    # Compare with your implemented rdm4
    if not np.isclose(rdm4[i, j, k, l, m, n, p, q], expected):
        print(f"4RDM mismatch at ({i},{j},{k},{l},{m},{n},{p},{q}): "
              f"Expected {expected}, Got {rdm4[i, j, k, l, m, n, p, q]}")
    if not np.isclose(rdm4_rotated[i, j, k, l, m, n, p, q], expected_rotated):
            print(f"Rotated 4RDM mismatch at ({i},{j},{k},{l},{m},{n},{p},{q}): "
                f"Expected {expected_rotated}, Got {rdm4_rotated[i, j, k, l, m, n, p, q]}")
"""

'\n# --- Verify 4RDM; skip, takes 15 min---\norbitals = [1, 2, 3, 4]\nfor i, j, k, l, m, n, p, q in itertools.product(orbitals, repeat=8):\n    expected = 0.0\n    expected_rotated = 0.0\n    # Sum over all spin combinations for the four creation operators\n    for spin_i in [cre_a, cre_b]:\n        for spin_j in [cre_a, cre_b]:\n            for spin_k in [cre_a, cre_b]:\n                for spin_l in [cre_a, cre_b]:\n                    # Determine the corresponding annihilation operators\n                    des_m = des_a if spin_l == cre_a else des_b\n                    des_n = des_a if spin_k == cre_a else des_b\n                    des_p = des_a if spin_j == cre_a else des_b\n                    des_q = des_a if spin_i == cre_a else des_b\n\n                    # Construct the operator: cre(i) cre(j) cre(k) cre(l) des(q) des(p) des(n) des(m)\n                    op = ffsim.linear_operator(\n                        ffsim.FermionOperator({\n                            (spin_i(i), s

In [8]:
# energy consistency check on rotations

E_rdms = np.einsum('pq,pq->', h1e, rdm1) + .5 * np.einsum('pqrs,prsq->', g2e_full, rdm2)
E_rdms_rotated = np.einsum('pq,pq->', h1e_rotated, rdm1_rotated) + .5 * np.einsum('pqrs,prsq->', g2e_full_rotated, rdm2_rotated)

# Construct the ffsim 1P Hamiltonian using chemist notation integrals
hamiltonian = ffsim.MolecularHamiltonian(
    one_body_tensor=h1e, 
    two_body_tensor=g2e_full, # here: chemist notation
    constant=0
)

# Interaction
hamiltonian_rotated = ffsim.MolecularHamiltonian(
    one_body_tensor=h1e_rotated, 
    two_body_tensor=g2e_full_rotated, # here: chemist notation
    constant=0
)

# Generate the linear operators acting on the (N_alpha, N_beta) subspace
h_linop = ffsim.linear_operator(hamiltonian, norb, nelec)
h_linop_rotated = ffsim.linear_operator(hamiltonian_rotated, norb, nelec)

E_fci = np.vdot(fci_state, h_linop @ fci_state)
E_fci_rotated = np.vdot(fci_state_rotated, h_linop_rotated @ fci_state_rotated)

assert np.allclose(E_fci, [E_rdms, E_rdms_rotated, E_fci_rotated]), "Some energies do not coincide."

In [9]:
# ============================================================================
# SECTION 3: Squared commutator norm 
# ============================================================================

# checked numerically, it's almost surely correct

$$
\text{squared commutator exp value}(h,g_{chem},D^{(1)},D^{(2)},D^{(3)},D^{(4)},C) = A + B + B^* + C
$$

with $g_{pqrs} = g_{chem, prqs}$ and

$$
A=
\sum_{t,t'\in C}\sum_{p,q}\sum_{p',q'}
h_{pq}\,h_{p'q'}\,
(-\delta_{pt}+\delta_{qt})\,(-\delta_{p't'}+\delta_{q't'})\,
\Bigl(
D^{(2)}_{p\,p'\,q'\,q}
+\delta_{q p'}\,D^{(1)}_{p\,q'}
\Bigr)
$$

$$
B = \frac12
\sum_{t,t'\in C}\sum_{p,q}\sum_{p',q',r',s'}
h_{pq}\,g_{p'q'r's'}\,
(-\delta_{pt}+\delta_{qt})\,
(-\delta_{p't'}-\delta_{q't'}+\delta_{s't'}+\delta_{r't'})\,
\Bigl[
D^{(3)}_{p\,p'\,q'\,s'\,r'\,q}
+\delta_{q p'}\,D^{(2)}_{p\,q'\,s'\,r'}
+\delta_{q q'}\,D^{(2)}_{p\,p'\,r'\,s'}
\Bigr]
$$

$$
C =
\frac14
\sum_{t,t'\in C}\sum_{p,q,r,s}\sum_{p',q',r',s'}
g_{pqrs}\,g_{p'q'r's'}\,
(-\delta_{pt}-\delta_{qt}+\delta_{st}+\delta_{rt})\,
(-\delta_{p't'}-\delta_{q't'}+\delta_{s't'}+\delta_{r't'})\,
\Bigl[
D^{(4)}_{p\,q\,p'\,q'\,s'\,r'\,s\,r}


+\delta_{s p'}\,D^{(3)}_{p\,q\,q'\,s'\,r'\,r}
+\delta_{s q'}\,D^{(3)}_{p\,q\,p'\,r'\,s'\,r}
+\delta_{r p'}\,D^{(3)}_{p\,q\,q'\,s'\,s\,r'}
+\delta_{r q'}\,D^{(3)}_{p\,q\,p'\,r'\,s\,s'}

+\delta_{r p'}\delta_{s q'}\,D^{(2)}_{p\,q\,s'\,r'}
+\delta_{r q'}\delta_{s p'}\,D^{(2)}_{p\,q\,r'\,s'}
\Bigr]
$$

In [10]:
import numpy as np
import jax.numpy as jnp

# new implementation corresponding to formula above here
from src.cluster_number_operators import squared_commutator_exp_value

# same thing but manual - for testing
from src.cluster_number_operators import squared_commutator_exp_value_expected

In [11]:
# ============================================================================
# SECTION 4: Test squared commutator norm functions against exact one.
# ============================================================================

clusters = [[0, 1, 2], [4, 6, 2], [1, 6, 2], [3], [5, 3], [0, 2, 4, 6], range(1), range(2), range(3), range(4), range(5), range(6), range(7)]

h1e_zeros = np.zeros((norb, norb))
g2e_full_zeros = np.zeros((norb, norb, norb, norb))

# Construct the ffsim 1P Hamiltonian using chemist notation integrals
hamiltonian1 = ffsim.MolecularHamiltonian(
    one_body_tensor=h1e, 
    two_body_tensor=g2e_full_zeros, # here: chemist notation
    constant=ecore
)

# Interaction
hamiltonian2 = ffsim.MolecularHamiltonian(
    one_body_tensor=h1e_zeros, 
    two_body_tensor=g2e_full, # here: chemist notation
    constant=ecore
)

# Generate the linear operators acting on the (N_alpha, N_beta) subspace
h1_linop = ffsim.linear_operator(hamiltonian1, norb, nelec)
h2_linop = ffsim.linear_operator(hamiltonian2, norb, nelec)

for cluster in clusters:
    A_rdm, B_rdm, C_rdm = squared_commutator_exp_value(h1e, g2e_full, rdm1, rdm2, rdm3, rdm4, cluster, return_terms=True)
    A_op, B_op, C_op = squared_commutator_exp_value_expected(h1_linop, h2_linop, fci_state, cluster, norb, nelec, return_terms=True)

    assert (A_rdm - A_op).round(10) == 0, f"A diff: {(A_rdm - A_op).round(10)}"
    assert (B_rdm - B_op).round(10) == 0, f"B diff: {(B_rdm - B_op).round(10)}"
    assert (C_rdm - C_op).round(10) == 0, f"C diff: {(C_rdm - C_op).round(10)}"

In [12]:
# ============================================================================
# SECTION 5: Build old and new cost functions
# ============================================================================

# new commutator-based cost functions; old one uses squared_commutator_exp_value, v2 its logic
from src.cluster_number_operators import number_commutator_cost_v1, number_commutator_cost_v2
from optimize_symmetries import commutator_cost # the old one

from src.cluster_number_operators import number_matrix_to_operators
import optimize_symmetries
optimize_symmetries.pyscf = pyscf # in case try-except import code in optimize_symmetries fails
optimize_symmetries.ffsim = ffsim # in case try-except import code in optimize_symmetries fails

number_operators = number_matrix_to_operators(cluster_matrix, norb, nelec) # no ghost

moldata = pyscf.lib.chkfile.load_mol(mf.chkfile)
mf_update = pyscf.scf.RHF(mol)
mf_update.update_from_chk(mf.chkfile)
moldata_ffsim = ffsim.MolecularData.from_scf(mf_update)

f_old = commutator_cost(moldata_ffsim, number_operators, fci_state)
f_v1 = number_commutator_cost_v1(h1e, g2e_full, rdm1, rdm2, rdm3, rdm4, cluster_matrix, with_ghost=False)
f_v2 = number_commutator_cost_v2(h1e, g2e_full, rdm1, rdm2, rdm3, rdm4, cluster_matrix, with_ghost=False)

In [16]:
# ============================================================================
# SECTION 6: Compare
# ============================================================================

from math import comb

number_tests = 10
length_x = comb(norb, 2)
xs = [np.random.rand(length_x) for _ in range(number_tests)]
for x in xs:
    f_old_value = f_old(x)
    print("Computed f_old(x)")
    f_v1_value = f_v1(x)
    print("Computed f_v1(x)")
    f_v2_value = f_v2(x)
    print("Computed f_v2(x)")
    print()
    assert np.isclose(f_old_value, f_v1_value), f"The old and v1 cost functions differ by {np.abs(f_old_value - f_v1_value)}"
    assert np.isclose(f_old_value, f_v2_value), f"The old and v2 cost functions differ by {np.abs(f_old_value - f_v2_value)}"

Computed f_old(x)
Computed f_v1(x)
Computed f_v2(x)

Computed f_old(x)
Computed f_v1(x)
Computed f_v2(x)

Computed f_old(x)


KeyboardInterrupt: 